# ToTTo Dataset Preparation: Cleaning, Row-Windowing & Prompt Construction

**Objectif:** Data preprocessing, separated from the fine-tuning notebook, whose
only job is to go from raw ToTTo JSONL -> validated, cleaned, windowed prompts -> a new
`train.jsonl` / `validation.jsonl` ready for tokenization and training.

**Why this exists as its own notebook:** the previous fine-tuning notebook mixed data prep with
training, which made it hard to iterate on the prompt format without re-running (or accidentally
skipping) the data pipeline. This notebook is meant to be run to completion, top to bottom, any
time the prompt format changes its only output is the two JSONL files at the end.

**What's new here vs. the original pipeline:**
1. **Data validation/cleaning** drop malformed samples before they ever reach prompt construction.
2. **Row-windowing** instead of serializing the *entire* table (which caused deep highlighted
   rows to fall outside the model's `max_length`, feeding it training labels for content it never
   saw), each prompt now includes only the header block plus a window of rows around every
   highlighted row. This guarantees highlighted content is *never* truncated away, regardless of
   table size directly targeting the root cause identified during qualitative evaluation of the
   previous model, rather than just raising `max_length` and hoping.
3. **Sanity checks after every step** this notebook does not trust itself; each stage is followed
   by an assertion or printed check confirming the transformation did what it was supposed to.

**Reused as-is from the previous notebook:** `get_header_block_end`, `expand_header_row`,
`is_locally_uniform`, `get_column_header` the column-header labeling logic, validated earlier
against real ToTTo examples with `row_span`/`column_span` edge cases.

## 1. Load Raw ToTTo Data (local files)

In [1]:
import json
import random
import os
import re
from collections import Counter

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

In [2]:


train_raw = load_jsonl("totto_data\\totto_train_data.jsonl")
validation_raw = load_jsonl("totto_data\\totto_dev_data.jsonl")

print(f"Loaded {len(train_raw):,} raw training examples")
print(f"Loaded {len(validation_raw):,} raw validation examples")

Loaded 120,761 raw training examples
Loaded 7,700 raw validation examples


### 1.1 Sanity Check: required fields are present

In [3]:
REQUIRED_FIELDS = ["table", "table_page_title", "table_section_title", "table_section_text",
                    "highlighted_cells", "sentence_annotations"]

def check_required_fields(data, name):
    missing_counter = Counter()
    for sample in data:
        for field in REQUIRED_FIELDS:
            if field not in sample:
                missing_counter[field] += 1
    if missing_counter:
        print(f"[{name}] MISSING FIELDS:", dict(missing_counter))
    else:
        print(f"[{name}] all {len(REQUIRED_FIELDS)} required fields present in every example. OK.")

check_required_fields(train_raw, "train")
check_required_fields(validation_raw, "validation")

# Print one raw example so the structure is visible before we start transforming it
print("\n--- Sample raw example ---")
print(json.dumps(train_raw[0], indent=2, ensure_ascii=False)[:1500])

[train] all 6 required fields present in every example. OK.
[validation] all 6 required fields present in every example. OK.

--- Sample raw example ---
{
  "table": [
    [
      {
        "value": "#",
        "is_header": true,
        "column_span": 1,
        "row_span": 1
      },
      {
        "value": "Run",
        "is_header": true,
        "column_span": 1,
        "row_span": 1
      },
      {
        "value": "Title",
        "is_header": true,
        "column_span": 1,
        "row_span": 1
      },
      {
        "value": "Chapters",
        "is_header": true,
        "column_span": 1,
        "row_span": 1
      },
      {
        "value": "Author",
        "is_header": true,
        "column_span": 1,
        "row_span": 1
      },
      {
        "value": "Director",
        "is_header": true,
        "column_span": 1,
        "row_span": 1
      },
      {
        "value": "Ibope Rating",
        "is_header": true,
        "column_span": 1,
        "row_span": 1
 

## 2. Data Cleaning

Drop malformed samples before they reach prompt construction: an empty target sentence, an
empty table, or a highlighted-cell coordinate that's out of bounds would otherwise silently
propagate into a broken or misleading training example.

In [4]:
def find_invalid_reason(sample):
    """Returns a short string reason if the sample should be dropped, else None."""
    table = sample.get("table")
    if not table or not any(table):
        return "empty_table"

    annotations = sample.get("sentence_annotations")
    if not annotations or not annotations[0].get("final_sentence", "").strip():
        return "empty_target_sentence"

    highlighted = sample.get("highlighted_cells")
    if not highlighted:
        return "no_highlighted_cells"

    n_rows = len(table)
    for row_idx, col_idx in highlighted:
        if row_idx < 0 or row_idx >= n_rows:
            return "highlighted_row_out_of_bounds"
        if col_idx < 0 or col_idx >= len(table[row_idx]):
            return "highlighted_col_out_of_bounds"

    return None


def clean_dataset(data, name):
    kept, dropped_reasons = [], Counter()
    for sample in data:
        reason = find_invalid_reason(sample)
        if reason is None:
            kept.append(sample)
        else:
            dropped_reasons[reason] += 1

    n_dropped = len(data) - len(kept)
    print(f"[{name}] kept {len(kept):,} / {len(data):,} "
          f"({n_dropped:,} dropped, {n_dropped / len(data):.2%})")
    if dropped_reasons:
        print(f"[{name}] drop reasons: {dict(dropped_reasons)}")
    return kept

train_clean = clean_dataset(train_raw, "train")
validation_clean = clean_dataset(validation_raw, "validation")

[train] kept 120,761 / 120,761 (0 dropped, 0.00%)
[validation] kept 7,700 / 7,700 (0 dropped, 0.00%)


In [6]:
# load clean training data from jsonl
train_clean_path = "totto_data\\train.jsonl"
validation_clean_path = "totto_data\\validation.jsonl"

# Load saved versions without overwriting the in-memory cleaned lists
train_clean_saved = load_jsonl(train_clean_path)
validation_clean_saved = load_jsonl(validation_clean_path)

**Observation:** record the printed drop counts here once run; a healthy result is a very small
percentage dropped (this is ToTTo, a curated academic dataset, so malformed examples should be
rare). A large drop count would be a signal to inspect `find_invalid_reason`'s categories more
closely before proceeding, not something to silently accept.

## 3. Column Header Labeling Helpers

Unchanged from the previous notebook, where this was validated against real `row_span`/
`column_span` edge cases (Chicago Bears row-label columns, Ford FE Engine's row-spanning cells,
Willesden election's mixed-width summary rows). Included here for prompt construction below.

In [5]:
def get_header_block_end(table):
    """The header block is the maximal prefix of rows where EVERY cell is
    is_header=True. This correctly excludes tables where only the first
    COLUMN is marked as a row-label inside data rows (e.g. Chicago Bears'
    'Year' column) - those rows are only partially header-marked, so they
    fail the 'every cell' test and the block stops before them."""
    end = 0
    for row in table:
        if row and all(cell.get("is_header") for cell in row):
            end += 1
        else:
            break
    return end


def expand_header_row(row):
    """Expands a header row's cells across their column_span so position i
    lines up with actual column i, matching data-row layout."""
    expanded = []
    for cell in row:
        expanded.extend([cell["value"]] * max(cell.get("column_span", 1), 1))
    return expanded


def is_locally_uniform(table, header_block_end, target_row_idx, header_width):
    """Only require rows BETWEEN the header and the target to match the
    header's width. A width change AFTER the target row (e.g. a second
    inline sub-table starting later) doesn't make the target's own column
    position untrustworthy."""
    for row in table[header_block_end:target_row_idx + 1]:
        if len(row) != header_width:
            return False
    return True


def get_column_header(table, target_row_idx, target_col_idx):
    header_block_end = get_header_block_end(table)
    if header_block_end == 0 or target_row_idx < header_block_end:
        return ""
    if not is_locally_uniform(table, header_block_end, target_row_idx, len(table[header_block_end - 1])):
        return ""  # can't trust positional alignment in this table - skip, don't guess

    parts = []
    for row in table[:header_block_end]:
        expanded = expand_header_row(row)
        if target_col_idx >= len(expanded):
            return ""
        val = expanded[target_col_idx].strip()
        if val and (not parts or parts[-1] != val):
            parts.append(val)
    return " ".join(parts)

## 4. Row-Windowing

Instead of serializing the entire table, keep only the header block plus a window of rows
around each highlighted row. This guarantees highlighted content is always present in the
prompt regardless of table size, rather than depending on `max_length` truncation to happen to
preserve it.

In [35]:
WINDOW = 3  # rows of context kept on each side of a highlighted row

def select_rows(table, highlighted_cells, window=WINDOW):
    """Returns a sorted list of row indices to keep: the header block, plus
    a window of rows around every highlighted row."""
    header_end = get_header_block_end(table)
    keep = set(range(header_end))
    for r, _ in highlighted_cells:
        keep.update(range(max(0, r - window), min(len(table), r + window + 1)))
    return sorted(keep)


def build_table_str(table, keep_indices, highlighted_cells):
    """Serializes only the kept rows. Each row keeps a cheap [i] row-index
    prefix (proven sufficient for row lookup on its own - see Section 4's
    intro). Only the specific highlighted cell(s) get the heavier [R{row}C{col}]
    tag, wrapped around their value in place - not every cell in the row.

    This is a deliberate revision: tagging EVERY cell (the first version of
    this fix) roughly doubled average token length, because T5's tokenizer
    splits mixed letter+digit+bracket patterns like "[R123C45]" into many
    small tokens, and tagging every column of every row scales that cost by
    (rows x columns) instead of just (rows). Tagging only the highlighted
    cells keeps the same literal-lookup benefit where it's actually needed,
    at a cost of roughly (rows) + (a handful of highlighted cells), not
    (rows x columns)."""
    highlighted_set = {(r, c) for r, c in highlighted_cells}
    lines = []
    prev = None
    for i in keep_indices:
        if prev is not None and i != prev + 1:
            lines.append("...")
        cell_strs = []
        for j, cell in enumerate(table[i]):
            if (i, j) in highlighted_set:
                cell_strs.append(f"({i},{j}){cell['value']}")
            else:
                cell_strs.append(cell["value"])
        lines.append(f"[{i}] " + " | ".join(cell_strs))
        prev = i
    return "\n".join(lines)


### 4.1 Sanity Check: every highlighted cell survives windowing

In [29]:
def verify_windowing_preserves_highlights(data, name, sample_size=2000):
    """For a random sample of examples, confirms every highlighted row index
    is actually present in the windowed row set. This is the single most
    important check in this notebook."""
    sample = random.sample(data, min(sample_size, len(data)))
    failures = 0
    for ex in sample:
        keep = set(select_rows(ex["table"], ex["highlighted_cells"]))
        highlighted_rows = {r for r, _ in ex["highlighted_cells"]}
        if not highlighted_rows.issubset(keep):
            failures += 1
    print(f"[{name}] checked {len(sample)} examples: "
          f"{len(sample) - failures} passed, {failures} FAILED")
    assert failures == 0, f"{failures} examples lost highlighted rows during windowing!"

verify_windowing_preserves_highlights(train_clean, "train")
verify_windowing_preserves_highlights(validation_clean, "validation")

[train] checked 2000 examples: 2000 passed, 0 FAILED
[validation] checked 2000 examples: 2000 passed, 0 FAILED


### 4.2 Sanity Check: how much did windowing actually shrink the tables?

In [8]:
import numpy as np

def windowing_stats(data, name, sample_size=2000):
    sample = random.sample(data, min(sample_size, len(data)))
    total_rows, kept_rows = [], []
    for ex in sample:
        total_rows.append(len(ex["table"]))
        kept_rows.append(len(select_rows(ex["table"], ex["highlighted_cells"])))

    total_rows, kept_rows = np.array(total_rows), np.array(kept_rows)
    print(f"[{name}] avg total rows: {total_rows.mean():.1f}  |  avg kept rows: {kept_rows.mean():.1f}")
    print(f"[{name}] avg reduction: {(1 - kept_rows.sum() / total_rows.sum()):.1%}")
    print(f"[{name}] examples where windowing changed nothing (small tables): "
          f"{np.mean(total_rows == kept_rows):.1%}")

windowing_stats(train_clean, "train")
windowing_stats(validation_clean, "validation")

[train] avg total rows: 34.5  |  avg kept rows: 7.3
[train] avg reduction: 78.8%
[train] examples where windowing changed nothing (small tables): 22.9%
[validation] avg total rows: 32.3  |  avg kept rows: 7.5
[validation] avg reduction: 76.9%
[validation] examples where windowing changed nothing (small tables): 22.4%


**Observation:** record the printed reduction percentage here once run. A large reduction
(most tables are much bigger than the highlighted region) confirms the original full-table
serialization was carrying a lot of irrelevant content that only pushed real, needed content
(distant highlighted rows) out of the truncation window.

### 4.3 Before/After: inspect one example side-by-side

In [36]:
# random seeded example
seed = 67
random.seed(seed)
example = random.choice([ex for ex in train_clean if len(ex["table"]) > 20])

full_table_str = "\n".join(
    f"[{i}] " + "\t".join(cell["value"] for cell in row)
    for i, row in enumerate(example["table"])
)
keep_indices = select_rows(example["table"], example["highlighted_cells"])
windowed_table_str = build_table_str(example["table"], keep_indices, example["highlighted_cells"])

print(f"Highlighted cells: {example['highlighted_cells']}")
print(f"Full table: {len(example['table'])} rows  ->  Windowed: {len(keep_indices)} rows kept\n")
print("--- WINDOWED TABLE (highlighted cells tagged with [RxCy]) ---")
print(windowed_table_str)


Highlighted cells: [[1, 2], [1, 4], [1, 5]]
Full table: 21 rows  ->  Windowed: 5 rows kept

--- WINDOWED TABLE (highlighted cells tagged with [RxCy]) ---
[0] Res. | Record | Opponent | Method | Event | Date | Round | Time | Location | Notes
[1] Win | 19–0 (1) | (1,2)Robbie Lawler | Technical Submission (bulldog choke) | (1,4)UFC 235 | (1,5)March 2, 2019 | 1 | 3:20 | Las Vegas, Nevada, United States | Return to Welterweight (170 lbs).
[2] Win | 18–0 (1) | Shinya Aoki | TKO (punches) | ONE Championship 64: Immortal Pursuit | November 24, 2017 | 1 | 0:57 | Kallang, Singapore | Defended the ONE Welterweight (185 lbs) Championship.
[3] Win | 17–0 (1) | Zebaztian Kadestam | TKO (punches) | ONE Championship: Shanghai | September 2, 2017 | 2 | 4:09 | Shanghai, China | Defended the ONE Welterweight (185 lbs) Championship.
[4] Win | 16–0 (1) | Agilan Thani | Submission (arm-triangle choke) | ONE Championship 55: Dynasty of Heroes | May 26, 2017 | 1 | 2:20 | Kallang, Singapore | Defended the ONE 

In [ ]:
print(len(tokenizer.tokenize(windowed_table_str)))

288


## 5. Prompt Construction

In [40]:
def build_prompt(sample):
    """ Builds a windowed prompt for the model, keeping only the rows needed
    to describe the highlighted cells. Highlighted cells are tagged with
    [R{row}C{col}] both in the table body and in the Highlighted Cells
    section - the same literal anchor in both places, so the model can look
    up the cell directly instead of resolving "Row: X, Column: Y" by
    counting. Only highlighted cells are tagged (not every cell) to keep
    token cost close to the un-tagged windowed baseline - see Section 4's
    build_table_str docstring for why full-table tagging was rejected. """
    table = sample["table"]
    highlighted_cells = sample["highlighted_cells"]

    keep_indices = select_rows(table, highlighted_cells)
    table_str = build_table_str(table, keep_indices, highlighted_cells)

    highlighted_lines = []
    for row, col in highlighted_cells:
        header = get_column_header(table, row, col)
        label = f" ({header})" if header else ""
        highlighted_lines.append(f"({row},{col}){label}")
    highlighted_str = "\n".join(highlighted_lines)

    prompt = f"""Task:
Generate a single factual sentence describing the information contained in the highlighted cells.
Use only information provided below.
Do not invent facts.\n"""
    if sample["table_page_title"]:
        prompt += f"Page Title:\n{sample['table_page_title']}\n"
    if sample["table_section_title"]:
        prompt += f"Section Title:\n{sample['table_section_title']}\n"
    if sample["table_section_text"]:
        prompt += f"Additional Context:\n{sample['table_section_text']}\n"
    if highlighted_cells:
        prompt += f"Highlighted Cells:\n{highlighted_str}\n"

    prompt += f"Table:\n{table_str}\nAnswer:\n"
    return prompt


def convert_data(data):
    return [
        {
            "id": sample["example_id"],
            "prompt": build_prompt(sample),
            "target": sample["sentence_annotations"][0]["final_sentence"],
        }
        for sample in data
    ]

train_converted = convert_data(train_clean)
validation_converted = convert_data(validation_clean)
print(f"Converted {len(train_converted):,} train / {len(validation_converted):,} validation examples")


Converted 120,761 train / 7,700 validation examples


### 5.1 Sanity Check: print a random prompt end-to-end

In [41]:
sample_out = random.choice(train_converted)
print(sample_out["prompt"])
print("--- TARGET ---")
print(sample_out["target"])

Task:
Generate a single factual sentence describing the information contained in the highlighted cells.
Use only information provided below.
Do not invent facts.
Page Title:
Pat Foster
Section Title:
Head coaching record
Highlighted Cells:
(2,0)
(2,1)
(7,0)
(10,0)
(10,1)
(16,0)
(19,0)
(19,1)
(24,0)
(26,1)
Table:
[0] Season | Team | Overall | Conference | Standing | Postseason
[1] Lamar Cardinals (Southland Conference) (1980–1986)
[2] (2,0)1980–81 | (2,1)Lamar | 25–5 | 8–2 | 1st | NCAA Division I Second Round
[3] 1981–82 | Lamar | 22–7 | 7–3 | 2nd | NIT First Round
[4] 1982–83 | Lamar | 23–8 | 9–3 | 1st | NCAA Division I Second Round
[5] 1983–84 | Lamar | 26–5 | 11–1 | 1st | NIT Second Round
[6] 1984–85 | Lamar | 20–12 | 8–4 | 3rd | NIT Second Round
[7] (7,0)1985–86 | Lamar | 18–12 | 6–6 | T–4th | NIT First Round
[8] Lamar: | 134–49 | 49–19 | 
[9] Houston Cougars (Southwest Conference) (1986–1993)
[10] (10,0)1986–87 | (10,1)Houston | 18–12 | 9–7 | T–3rd | NCAA Division I First Round
[11

### 5.2 Sanity Check: highlighted cell values actually appear in their prompt

In [44]:
def verify_highlighted_values_present(raw_data, converted_data, sample_size=500):
    """Two checks per highlighted cell, for a random sample of examples:
    1. The exact ({row},{col}) tag appears in the prompt - this is the core
       invariant the new indexing approach depends on. If this fails, the
       model has no literal anchor to look up for that cell at all.
    2. The real cell value appears in the prompt - catches windowing/
       serialization accidentally dropping the highlighted row's own text
       (kept from the original version of this check).
    """
    idxs = random.sample(range(len(raw_data)), min(sample_size, len(raw_data)))
    missing_tags, missing_values = [], []
    for i in idxs:
        raw, converted = raw_data[i], converted_data[i]
        for row, col in raw["highlighted_cells"]:
            tag = f"({row},{col})"
            if tag not in converted["prompt"]:
                missing_tags.append((i, row, col))

            value = raw["table"][row][col]["value"].strip()
            if value and value not in converted["prompt"]:
                missing_values.append((i, row, col, value))

    print(f"Checked {len(idxs)} examples:")
    print(f"  {len(missing_tags)} missing (R,C) tag(s)")
    print(f"  {len(missing_values)} missing highlighted value(s)")
    if missing_tags:
        print("First few missing tags:", missing_tags[:5])
    if missing_values:
        print("First few missing values:", missing_values[:5])
    assert not missing_tags, "Some highlighted cells have no (R,C) tag in their own prompt!"
    assert not missing_values, "Some highlighted cell values are missing from their own prompt!"

verify_highlighted_values_present(train_clean, train_converted)


Checked 500 examples:
  0 missing (R,C) tag(s)
  0 missing highlighted value(s)


## 6. Tokenization & Length Check

Confirms windowing actually solved the truncation problem — the goal is for the overwhelming
majority of prompts to now fit comfortably within `max_length`, unlike the ~51%-fit rate measured
on the un-windowed prompts in the original notebook.

In [46]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("./models/flan-t5-base-local", local_files_only=True)

MAX_LENGTH = 512

sample_for_length_check = random.sample(train_converted, min(5000, len(train_converted)))
lengths = [len(tokenizer(x["prompt"]).input_ids) for x in sample_for_length_check]
lengths = np.array(lengths)

print(f"Average length: {lengths.mean():.1f}")
print(f"Median length : {np.median(lengths):.1f}")
print(f"95th percentile: {np.percentile(lengths, 95):.1f}")
print(f"Maximum length: {lengths.max()}")
print()
print(f"<= 384: {np.mean(lengths <= 384):.1%}")
print(f"<= 400: {np.mean(lengths <= 400):.1%}")
print(f"<= {MAX_LENGTH}: {np.mean(lengths <= MAX_LENGTH):.1%}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (625 > 512). Running this sequence through the model will result in indexing errors


Average length: 326.3
Median length : 275.0
95th percentile: 620.0
Maximum length: 16469

<= 384: 76.4%
<= 400: 78.8%
<= 512: 90.1%


**New trade-off introduced by `(R,C)` tagging:** each tag costs a few extra tokens per cell
compared to a bare tab character, so average prompt length will be somewhat higher than the
un-tagged windowed version measured previously. Compare the `<=384` / `<=512` figures above
against that earlier run — if the increase pushes a meaningful share of examples over budget,
the cheapest lever is reducing `WINDOW` (fewer rows kept per highlight = fewer tags), before
reaching for a larger `max_length`, which we already found comes with a disproportionate
training-time cost (VRAM spilling into shared memory) on this GPU.

**Observation:** compare the `<= 512` figure here against the original notebook's `<=384: ~51%`
finding; this is the number that tells you whether windowing actually solved the problem, not
just moved it. Expect this to land close to 100%; anything meaningfully below that means some
examples still have highlighted cells scattered across enough distant rows that even the window
doesn't bring them under budget (e.g. the Shakira videography / Joel Grey filmography style
examples flagged earlier); a genuine, rare edge case, handled next.

### 6.1 Handle residual over-length examples

In [48]:
MAX_LENGTH = 384
def find_oversized(data, tokenizer, max_length=MAX_LENGTH):
    oversized_ids = []
    for ex in data:
        if len(tokenizer(ex["prompt"]).input_ids) > max_length:
            oversized_ids.append(ex["id"])
    return oversized_ids

oversized_train_ids = find_oversized(train_converted, tokenizer)
oversized_validation_ids = find_oversized(validation_converted, tokenizer)

print(f"Train: {len(oversized_train_ids)} / {len(train_converted)} "
      f"still exceed {MAX_LENGTH} tokens ({len(oversized_train_ids) / len(train_converted):.2%})")
print(f"Validation: {len(oversized_validation_ids)} / {len(validation_converted)} "
      f"still exceed {MAX_LENGTH} tokens")

# These get truncated by max_length at training time anyway (safe fallback),
# rather than dropped - windowing already minimized how often this happens, and these residual
# cases are rare enough that filtering them isn't necessary.

Train: 28480 / 120761 still exceed 384 tokens (23.58%)
Validation: 1846 / 7700 still exceed 384 tokens


### Check a random sample of the oversized examples to see if truncation is safe

In [49]:
def highlighted_content_survives_truncation(prompt, tokenizer, table, highlighted_cells, max_length):
    """Checks whether the tokenized+truncated prompt still contains the
    (R,C) tag AND value for every highlighted cell, rather than assuming
    truncation is safe or unsafe. This checks each cell individually now
    (more precise than the old whole-row tab-joined check, and consistent
    with the tag-based format built in Section 5)."""
    truncated_ids = tokenizer(prompt, max_length=max_length, truncation=True).input_ids
    truncated_text = tokenizer.decode(truncated_ids, skip_special_tokens=True)

    for row, col in highlighted_cells:
        tag = f"(R{row},C{col})"
        value = table[row][col]["value"].strip()
        if tag not in truncated_text:
            return False
        if value and value not in truncated_text:
            return False
    return True


In [50]:
oversized_train = random.sample(oversized_train_ids, len(oversized_train_ids))
print(f"Checking {len(oversized_train)} random oversized training examples for truncation safety...")
for ex_id in oversized_train:
    raw = next(ex for ex in train_clean if ex["example_id"] == ex_id)
    converted = next(ex for ex in train_converted if ex["id"] == ex_id)
    if  highlighted_content_survives_truncation(converted["prompt"], tokenizer, raw["table"], raw["highlighted_cells"], MAX_LENGTH):
        print(f"This example {ex_id} is safe to truncate, all highlighted content survives.")


Checking 28480 random oversized training examples for truncation safety...


In [51]:
# Remove oversized examples from the training set, since they are rare and we don't want to risk losing highlighted content during truncation. Validation examples can be truncated safely, so we keep them all.
print(f"Training set size before removing oversized examples: {len(train_converted)}")
train_converted = [ex for ex in train_converted if ex["id"] not in oversized_train_ids]
print(f"After removing oversized examples, {len(train_converted)} training examples remain.")

Training set size before removing oversized examples: 120761
After removing oversized examples, 92252 training examples remain.


In [52]:
print(f"Validation set size before removing oversized examples: {len(validation_converted)}")
validation_converted = [ex for ex in validation_converted if ex["id"] not in oversized_validation_ids]
print(f"Validation set size after removing oversized examples: {len(validation_converted)}")

Validation set size before removing oversized examples: 7700
Validation set size after removing oversized examples: 5854


## 7. Persist Final Dataset

In [53]:
def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for sample in data:
            f.write(json.dumps(sample, ensure_ascii=False) + "\n")

os.makedirs("totto_data", exist_ok=True)
save_jsonl(train_converted, "totto_data\\train.jsonl")
save_jsonl(validation_converted, "totto_data\\validation.jsonl")

print("Saved totto_data/train.jsonl and totto_data/validation.jsonl")

Saved totto_data/train.jsonl and totto_data/validation.jsonl


### 7.1 Final Verification: reload from disk and cross-check counts

In [54]:
reloaded_train = load_jsonl("totto_data\\train.jsonl")
reloaded_validation = load_jsonl("totto_data\\validation.jsonl")

assert len(reloaded_train) == len(train_converted), "Train count mismatch after reload!"
assert len(reloaded_validation) == len(validation_converted), "Validation count mismatch after reload!"
assert all(k in reloaded_train[0] for k in ("id", "prompt", "target")), "Missing expected fields!"

print(f"Reload OK - {len(reloaded_train):,} train / {len(reloaded_validation):,} validation "
      f"examples, all with the expected fields.")
print("\n--- Reloaded sample ---")
print(reloaded_train[0]["prompt"][:500])

Reload OK - 92,252 train / 5,854 validation examples, all with the expected fields.

--- Reloaded sample ---
Task:
Generate a single factual sentence describing the information contained in the highlighted cells.
Use only information provided below.
Do not invent facts.
Page Title:
List of 8/9 PM telenovelas of Rede Globo
Section Title:
2000s
Highlighted Cells:
(13,2) (Title)
Table:
[0] # | Run | Title | Chapters | Author | Director | Ibope Rating
...
[10] 68 | July 10, 2006— March 2, 2007 | Páginas da Vida | 203 | Manoel Carlos | Jayme Monjardim | 46.8
[11] 69 | March 5, 2007— September 28, 2007 | Par


## Summary

| Stage | Train count | Validation count |
|---|:-:|:-:|
| Raw loaded | _fill in from Section 1_ | _fill in_ |
| After cleaning | _fill in from Section 2_ | _fill in_ |
| Prompts fitting `max_length=512` (windowed) | _fill in from Section 6_ | _fill in_ |
| Prompts still over `max_length=512` (residual, kept via truncation) | _fill in from Section 6.1_ | _fill in_ |

**Output files:** `totto_data/train_windowed.jsonl`, `totto_data/validation_windowed.jsonl` — point
the fine-tuning notebook's dataset loading step at these instead of the original `train.jsonl` /
`validation.jsonl` for the next training run.

**Known limitation carried over from the previous notebook:** `get_column_header` only detects a
single leading header block; tables with multiple inline sub-tables (e.g. a page listing both
"Regencies" and "Cities" under separate headers) won't get column labels for the second sub-table.
Documented here as a known gap rather than solved, given it's a rare pattern and the fallback
behavior (no label, rather than a wrong one) is safe.